# 파이썬 심화 문제집 - 클래스

## 클래스 심화 (441 ~ 450)

### 441 클래스 - 간단한 ORM 쿼리빌더

SQL 쿼리를 메서드 체이닝으로 만드는 QueryBuilder를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class QueryBuilder:
    def __init__(self,table): self._table=table; self._conditions=[]; self._limit=None; self._order=None
    def where(self,**kwargs):
        for k,v in kwargs.items(): self._conditions.append(f"{k}='{v}'")
        return self
    def order_by(self,col,desc=False):
        self._order=f'{col} {"DESC" if desc else "ASC"}'; return self
    def limit(self,n): self._limit=n; return self
    def build(self):
        sql=f'SELECT * FROM {self._table}'
        if self._conditions: sql+=' WHERE '+' AND '.join(self._conditions)
        if self._order: sql+=f' ORDER BY {self._order}'
        if self._limit: sql+=f' LIMIT {self._limit}'
        return sql

q=(QueryBuilder('users')
    .where(city='서울',age='25')
    .order_by('name')
    .limit(10)
    .build())
print(q)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 442 클래스 - 간단한 테스트 프레임워크

assert를 활용한 간단한 단위 테스트 프레임워크를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class TestCase:
    _pass=0; _fail=0
    def assertEqual(self,a,b,msg=''):
        if a==b: TestCase._pass+=1; print(f'  ✅ PASS')
        else: TestCase._fail+=1; print(f'  ❌ FAIL: {a} != {b} {msg}')
    def assertTrue(self,expr,msg=''):
        self.assertEqual(bool(expr),True,msg)
    def run(self):
        for name in [m for m in dir(self) if m.startswith('test_')]:
            print(f'[{name}]')
            getattr(self,name)()
        print(f'결과: {TestCase._pass}통과 / {TestCase._fail}실패')

class MyTests(TestCase):
    def test_add(self): self.assertEqual(1+1,2)
    def test_str(self): self.assertEqual('ab'+'c','abc')
    def test_fail(self): self.assertEqual(1+1,3)

MyTests().run()
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 443 클래스 - 타입 검사 데코레이터

함수 파라미터 타입을 자동으로 검사하는 클래스 데코레이터를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class TypeCheck:
    def __init__(self,**types): self.types=types
    def __call__(self,func):
        def wrapper(**kwargs):
            for k,v in kwargs.items():
                if k in self.types and not isinstance(v,self.types[k]):
                    raise TypeError(f'{k}는 {self.types[k].__name__} 타입이어야 합니다')
            return func(**kwargs)
        return wrapper

@TypeCheck(name=str,age=int)
def create_user(name,age): print(f'생성: {name}({age}세)')

create_user(name='김파이',age=25)  # 정상
try:
    create_user(name='김파이',age='25')  # 오류
except TypeError as e: print(e)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 444 클래스 - 지연 초기화 (Lazy Loading)

속성을 처음 접근할 때만 계산하는 lazy property를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class LazyProperty:
    def __init__(self,func): self.func=func; self.name='_lazy_'+func.__name__
    def __get__(self,obj,cls):
        if obj is None: return self
        if not hasattr(obj,self.name):
            print(f'{self.func.__name__} 계산 중...')
            setattr(obj,self.name,self.func(obj))
        return getattr(obj,self.name)

class Circle:
    def __init__(self,r): self.r=r
    @LazyProperty
    def area(self): return 3.14159*self.r**2
    @LazyProperty
    def circumference(self): return 2*3.14159*self.r

c=Circle(5)
print(c.area)          # 계산 후 반환
print(c.area)          # 캐시에서 반환 (재계산 없음)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 445 클래스 - 함수형 프로그래밍 도구

map, filter, reduce를 지원하는 FunctionalList 클래스를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
from functools import reduce

class FList:
    def __init__(self,data): self.data=list(data)
    def map(self,f): return FList(map(f,self.data))
    def filter(self,f): return FList(filter(f,self.data))
    def reduce(self,f,init=None):
        if init is not None: return reduce(f,self.data,init)
        return reduce(f,self.data)
    def to_list(self): return self.data
    def __repr__(self): return f'FList({self.data})'

result=(FList(range(1,11))
    .filter(lambda x:x%2==0)
    .map(lambda x:x**2)
    .reduce(lambda a,b:a+b))
print(result)  # 2²+4²+6²+8²+10² = 220
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 446 클래스 - 간단한 OOP 시뮬레이션

은행 시스템 시뮬레이션: 여러 고객의 거래를 시뮬레이션하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
import random

class Bank:
    def __init__(self,name): self.name=name; self.accounts=[]
    def open_account(self,name,balance=0):
        acc={'name':name,'balance':balance,'transactions':[]}
        self.accounts.append(acc); return acc
    def deposit(self,acc,amount):
        acc['balance']+=amount
        acc['transactions'].append(('입금',amount))
    def withdraw(self,acc,amount):
        if acc['balance']>=amount:
            acc['balance']-=amount
            acc['transactions'].append(('출금',amount))
    def total_assets(self): return sum(a['balance'] for a in self.accounts)
    def report(self):
        print(f'=== {self.name} 보고서 ===')
        for a in self.accounts:
            print(f"{a['name']}: {a['balance']:,}원 (거래 {len(a['transactions'])}건)")
        print(f'총 자산: {self.total_assets():,}원')

b=Bank('파이썬은행')
kim=b.open_account('김파이',1000000)
lee=b.open_account('이파이',500000)
for _ in range(5): b.deposit(kim,random.randint(1000,50000))
for _ in range(3): b.withdraw(lee,random.randint(1000,30000))
b.report()
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 447 클래스 - 간단한 마크다운 파서

마크다운 형식의 텍스트를 HTML로 변환하는 파서를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
import re

class MarkdownParser:
    def parse(self,text):
        lines=text.strip().split('\n')
        html=[]
        for line in lines:
            if line.startswith('# '): html.append(f'<h1>{line[2:]}</h1>')
            elif line.startswith('## '): html.append(f'<h2>{line[3:]}</h2>')
            elif line.startswith('- '): html.append(f'<li>{line[2:]}</li>')
            elif line.startswith('**') and line.endswith('**'): html.append(f'<b>{line[2:-2]}</b>')
            elif line: html.append(f'<p>{line}</p>')
        return '\n'.join(html)

md='''
# 파이썬 소개
## 특징
- 읽기 쉬움
- 다목적 언어
**파이썬은 최고!**
'''
parser=MarkdownParser()
print(parser.parse(md))
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 448 클래스 - 간단한 의존성 주입

의존성 주입(DI) 컨테이너를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Container:
    def __init__(self): self._registry={}
    def register(self,name,cls,*args,**kwargs):
        self._registry[name]=(cls,args,kwargs)
    def resolve(self,name):
        cls,args,kwargs=self._registry[name]
        return cls(*args,**kwargs)

class EmailService:
    def send(self,to,msg): print(f'이메일→{to}: {msg}')

class SMSService:
    def send(self,to,msg): print(f'SMS→{to}: {msg}')

class UserService:
    def __init__(self,notifier): self.notifier=notifier
    def register(self,name,contact):
        print(f'{name} 가입 완료')
        self.notifier.send(contact, f'환영합니다, {name}님!')

container=Container()
container.register('email',EmailService)
container.register('sms',SMSService)
email=container.resolve('email')
userSvc=UserService(email)
userSvc.register('김파이','kim@example.com')
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 449 클래스 - 간단한 직렬화 프레임워크

클래스 인스턴스를 딕셔너리/JSON으로 자동 직렬화/역직렬화하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
import json

class Serializable:
    def to_dict(self): return {k:v for k,v in self.__dict__.items() if not k.startswith('_')}
    def to_json(self): return json.dumps(self.to_dict(), ensure_ascii=False)
    @classmethod
    def from_dict(cls,d):
        obj=cls.__new__(cls)
        for k,v in d.items(): setattr(obj,k,v)
        return obj
    @classmethod
    def from_json(cls,s): return cls.from_dict(json.loads(s))

class Product(Serializable):
    def __init__(self,name,price,stock):
        self.name=name; self.price=price; self.stock=stock
    def __repr__(self): return f'Product({self.name},{self.price},{self.stock})'

p=Product('사과',3000,100)
print(p.to_json())
p2=Product.from_json(p.to_json())
print(p2)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 450 클래스 종합 마무리 - 미니 게임 엔진

씬(Scene) 기반의 간단한 게임 엔진 구조를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Scene:
    def __init__(self,name): self.name=name; self.objects=[]
    def add(self,obj): self.objects.append(obj)
    def update(self): [o.update() for o in self.objects]
    def render(self): print(f'--- {self.name} ---'); [o.render() for o in self.objects]

class GameObject:
    def __init__(self,name,x=0,y=0): self.name=name; self.x=x; self.y=y
    def update(self): pass
    def render(self): print(f'  [{self.name}] at ({self.x},{self.y})')

class MovingObject(GameObject):
    def __init__(self,name,x,y,vx,vy):
        super().__init__(name,x,y); self.vx=vx; self.vy=vy
    def update(self): self.x+=self.vx; self.y+=self.vy

class GameEngine:
    def __init__(self): self.scenes={};  self.current=None
    def add_scene(self,scene): self.scenes[scene.name]=scene
    def load(self,name): self.current=self.scenes[name]; print(f'씬 로드: {name}')
    def loop(self,n=3):
        for i in range(n):
            print(f'\n[프레임 {i+1}]')
            self.current.update(); self.current.render()

engine=GameEngine()
scene=Scene('메인')
scene.add(MovingObject('영웅',0,0,1,0))
scene.add(MovingObject('몬스터',10,0,-1,0))
engine.add_scene(scene); engine.load('메인'); engine.loop(3)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요
